In [ ]:
import numpy as np
import time
import mujoco
import mujoco.viewer
import matplotlib.pyplot as plt

import osqp
import scipy.sparse as sp

In [ ]:
xml_path = '/home/frlab/mj_opt/xmls/systems/husky_ur5e/scene_husky_ur5e_3finger.xml'

model = mujoco.MjModel.from_xml_path(xml_path)
data = mujoco.MjData(model)

ee_site_name = "attachment_site"
ee_site_id = data.site(ee_site_name).id

In [ ]:
def draw_axes(user_scn, pos, rot_mat, size=0.1):
    R = rot_mat.reshape(3, 3)
    colors = [
        [1, 0, 0, 1],  # X - 빨강
        [0, 1, 0, 1],  # Y - 초록
        [0, 0, 1, 1],  # Z - 파랑
    ]
    for i in range(3):
        if user_scn.ngeom >= user_scn.maxgeom:
            break
        end = pos + R[:, i] * size
        mujoco.mjv_initGeom(
            user_scn.geoms[user_scn.ngeom],
            mujoco.mjtGeom.mjGEOM_ARROW,
            np.zeros(3), np.zeros(3), np.zeros(9),
            np.array(colors[i], dtype=np.float32)
        )
        mujoco.mjv_connector(
            user_scn.geoms[user_scn.ngeom],
            mujoco.mjtGeom.mjGEOM_ARROW, 0.005,
            pos, end
        )
        user_scn.ngeom += 1

#### 유틸리티 

In [ ]:
def so32Vec(so3mat):

    omega = np.array([so3mat[2, 1], so3mat[0, 2], so3mat[1, 0]])

    return omega


def Vec2so3(omega):
    w1, w2, w3 = np.array(omega).flatten()
    so3mat = np.array([[0, -w3, w2],
                    [w3, 0, -w1],
                    [-w2, w1, 0]])
    return so3mat


def MatrixLog3(R):
    cos_theta = np.clip((np.trace(R) - 1) / 2.0, -1, 1)
    theta = np.arccos(cos_theta)

    if theta < 1e-6:
        return np.zeros((3, 3))
    elif abs(theta - np.pi) < 1e-6:
        RpI = R + np.eye(3)
        col = np.argmax(np.linalg.norm(RpI, axis=0))
        w = RpI[:, col] / np.linalg.norm(RpI[:, col])
        return Vec2so3(w * theta)
    else:
        so3mat = theta / (2 * np.sin(theta)) * (R - np.array(R).T)
        return so3mat

#### 자코비안 분리하는 함수

In [ ]:
def seperate_jacobian(J_full):

    J_base = J_full[:, 0:6] # (6 x 6) 행렬
    J_wheel = J_full[:, 6:10] # (6 x 4) 행렬
    J_arm = J_full[:, 10:16] # (6 x 6) 행렬
    J_hand = J_full[:, 16:22] # (6 x 12) 행렬

    return J_base[:, [0, 5]], J_arm

In [ ]:
def build_wbc_qp(J_base_red, J_arm, V_desired,
                 w_task=1.0, w_reg_base=1e-2, w_reg_arm=1e-3,
                 dq_arm_max=1.5, v_base_max=0.5, w_base_max=1.0):
    # x = [v_x, w_z, dq_arm(6)]  → dim 8
    J = np.hstack([J_base_red, J_arm])           # (6, 8)
    W = w_task * np.eye(6)
    R = np.diag([w_reg_base, w_reg_base,
                 *([w_reg_arm]*6)])              # (8,8)

    P = J.T @ W @ J + R
    q = -J.T @ W @ V_desired

    # 박스 제약: l ≤ x ≤ u
    A = np.eye(8)
    lb = np.array([-v_base_max, -w_base_max, *([-dq_arm_max]*6)])
    ub = np.array([ v_base_max,  w_base_max, *([ dq_arm_max]*6)])

    prob = osqp.OSQP()
    prob.setup(sp.csc_matrix(P), q,
               sp.csc_matrix(A), lb, ub, verbose=False)
    return prob

def solve_wbc_qp(prob, P, q):
    prob.update(Px=sp.csc_matrix(P).data, q=q)   # warm-start 재사용
    res = prob.solve()
    return res.x   # [v_x, w_z, dq_arm(6)]


#### DLS 함수

In [ ]:
def DLS(twist_error, V_ff, J_b, lam, I, Kp):

    V_error = twist_error              # [lin, ang]
    V_desired = V_ff + Kp @ V_error    # (6,)

    JJt        = J_b @ J_b.T + lam * I
    desired_dq = J_b.T @ np.linalg.solve(JJt, V_desired)  # (6,)

    return desired_dq

#### 몸통의 선속도, 각속도를 바퀴로 분리하는 차륜 구동 함수

In [ ]:
def diff_drive_controller(L, mobile_ang, mobile_lin):

    # 지금은 mobile ang, lin이라고 적어놨는데 추후에 WBC QP를 풀어서 오는 desired_J_base? 맞나? 이걸 body_lin, body_ang을 받아서 휠로 변환

    v_left = (mobile_lin - mobile_ang * L/2) / r  # rad/s
    v_right = (mobile_lin + mobile_ang * L/2) / r  # rad/s

    return v_left, v_right, v_left, v_right

#### 종합 파라미터

In [ ]:
# DLS 파라미터
I = 2 * np.eye(6)
Kp = 5 * np.eye(6) # 70~80
V_des = 0
damping = 1e-3
V_ff = np.zeros(6)

# 모바일 파라미터 
L = 0.5708 # m
r = 0.1778 # m

# 이거 자체가 모바일 바디의 선속도
mobile_lin = 0.0
mobile_ang = 0.0

init_ee_pos = data.site(ee_site_id).xpos.copy()
init_ee_ori = data.site(ee_site_id).xmat.copy() 

dt = 0.005
sim_dt = model.opt.timestep

#### error twist 계산

In [ ]:
def calc_twist_error(current_pos, current_ori, target_pos, target_ori, init_ee_ori):

    R_current = current_ori.reshape(3, 3)
    pos_err = target_pos - current_pos

    R_tgt = init_ee_ori.reshape(3, 3)
    R_err = R_tgt @ R_current.T
    ori_err = so32Vec(MatrixLog3(R_err))

    return np.concatenate((pos_err, ori_err))


#### 시뮬레이션

In [ ]:
with mujoco.viewer.launch_passive(model, data, show_left_ui=False, show_right_ui=False) as viewer:
    # 패널 표시 설정
    viewer.opt.flags[mujoco.mjtVisFlag.mjVIS_CONTACTPOINT] = False

    sim_step = 0
    sim_time = 0.0
    duration = 1000.0
    step_start = time.time()

    mujoco.mj_resetData(model, data)
    mujoco.mj_forward(model, data)
    
    init_ee_pos = data.site(ee_site_id).xpos.copy()
    init_ee_ori = data.site(ee_site_id).xmat.copy()

    target_pos = init_ee_pos + np.array([-0.3, 0.0, 0.3])
    target_ori = init_ee_ori.copy()

    while viewer.is_running() and sim_time < duration:
        
        # manipulator
        current_pos = data.site(ee_site_id).xpos
        current_ori = data.site(ee_site_id).xmat
        J_full = np.zeros((6, model.nv))
        mujoco.mj_jacSite(model, data, J_full[:3], J_full[3:], ee_site_id)

        twist_error = calc_twist_error(current_pos, current_ori, target_pos, target_ori, init_ee_ori)
        J_body, J_arm = seperate_jacobian(J_full)
        desired_dq = DLS(twist_error, V_des, J_arm, damping, I, Kp)
        v_left, v_right, v_left, v_right = diff_drive_controller(L, mobile_ang, mobile_lin)
        
        current_q = data.qpos[11:17] # 팔
        desired_q = current_q + desired_dq * dt
        data.ctrl[4:10] = desired_q
        data.qfrc_applied[10:16] = data.qfrc_bias[10:16]
        data.ctrl[0:4] = [v_left, v_right, v_left, v_right]

        # hand
        data.ctrl[10:22] = np.zeros(12)

        mujoco.mj_step(model, data)
        sim_time += sim_dt
        sim_step += 1

        viewer.user_scn.ngeom = 0
        draw_axes(viewer.user_scn, target_pos, target_ori, size=0.2)
        draw_axes(viewer.user_scn, current_pos, current_ori.flatten(), size=0.2)
        viewer.sync()

        elapsed = time.time() - step_start
        remaining = sim_dt - elapsed

        # print log
        print(f"{elapsed*1000:.2f}ms")  # 한 스텝에 실제로 얼마나 걸리는지
        print(f"pos_err: {np.linalg.norm(twist_error):.3f}  dq_max: {np.max(np.abs(desired_dq)):.3f}") 

        if remaining > 0:
            time.sleep(remaining)
        step_start = time.time()